### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
# read all pdf inside a directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Python.pdf
Loaded 26 pages

Processing: SQL.pdf
Loaded 43 pages

Total documents loaded: 69


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-04-16T15:05:18+07:00', 'moddate': '2020-04-16T15:05:22+07:00', 'trapped': '/False', 'source': '..\\data\\pdf\\Python.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'Python.pdf', 'file_type': 'pdf'}, page_content='Python\nCheat Sheet\nPython 3 is a truly versatile programming language, loved \nboth by web developers, data scientists and software \nengineers. And there are several good reasons for that!\nOnce you get a hang of it, your development speed and productivity will soar!\n• Python is open-source and has a great support community, \n• Plus, extensive support libraries. \n• Its data structures are user-friendly.'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-04-16T15:05:18+07:00', 'moddate': '2020-04-16T15:05:22+07:00', 'trapped': '/False', 'source': '.

In [5]:
# text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    # split documents into smaller chunks for better RAG performance
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 69 documents into 105 chunks

Example chunk:
Content: Python
Cheat Sheet
Python 3 is a truly versatile programming language, loved 
both by web developers, data scientists and software 
engineers. And there are several good reasons for that!
Once you get...
Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-04-16T15:05:18+07:00', 'moddate': '2020-04-16T15:05:22+07:00', 'trapped': '/False', 'source': '..\\data\\pdf\\Python.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'Python.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-04-16T15:05:18+07:00', 'moddate': '2020-04-16T15:05:22+07:00', 'trapped': '/False', 'source': '..\\data\\pdf\\Python.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'Python.pdf', 'file_type': 'pdf'}, page_content='Python\nCheat Sheet\nPython 3 is a truly versatile programming language, loved \nboth by web developers, data scientists and software \nengineers. And there are several good reasons for that!\nOnce you get a hang of it, your development speed and productivity will soar!\n• Python is open-source and has a great support community, \n• Plus, extensive support libraries. \n• Its data structures are user-friendly.'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-04-16T15:05:18+07:00', 'moddate': '2020-04-16T15:05:22+07:00', 'trapped': '/False', 'source': '.

#### Embedding and VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    # handles document embeding generation using sentence transformer
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        generate embeddings for a list of texts
        Args:
            texts: list of text strings to embed
        
        Returns:
            numpy array of mebeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedings with shape: {embeddings.shape}")        
        return embeddings
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


d:\StudyAndWork\GenAI-AgenticAI\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Adarsh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 866.96it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Adarsh\AppData\Local\Temp\ipykernel_5560\3992575328.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


#### VectorStore

In [14]:
class VectorStore:
    """Managed document embeddings in a chromaDB vector string"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name (str, optional): _description_. Defaults to "pdf_documents".
            persist_directory (str, optional): _description_. Defaults to "../data/vector_store".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize chromadb client and collection"""
        try:
            # create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args: 
            documents: list of langchain documents
            embeddings: embeddings for documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)  
            
            documents_text.append(doc.page_content) 
            
            embeddings_list.append(embedding.tolist())    
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )     
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-04-16T15:05:18+07:00', 'moddate': '2020-04-16T15:05:22+07:00', 'trapped': '/False', 'source': '..\\data\\pdf\\Python.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'Python.pdf', 'file_type': 'pdf'}, page_content='Python\nCheat Sheet\nPython 3 is a truly versatile programming language, loved \nboth by web developers, data scientists and software \nengineers. And there are several good reasons for that!\nOnce you get a hang of it, your development speed and productivity will soar!\n• Python is open-source and has a great support community, \n• Plus, extensive support libraries. \n• Its data structures are user-friendly.'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-04-16T15:05:18+07:00', 'moddate': '2020-04-16T15:05:22+07:00', 'trapped': '/False', 'source': '.

In [16]:
# convert text to embeddings
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)

Generating embeding for 105 texts...


Batches: 100%|██████████| 4/4 [00:06<00:00,  1.60s/it]


Generated embedings with shape: (105, 384)
Adding 105 documents to vector store...
successfully added 105 documents to vector store
total documents in collection: 105


### Retriever Pipeline from VectorStore

In [17]:
class RAGRetriever:
    """handles query based retrieva; from vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        Args:
            vector_store: vector store containing document embeddings
            embedding_manager: manager for generating embedding for query
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for query
        
        Args: 
            query: search query
            top_k: number of top results to return
            score_threshold: minimum similarity
            
        returns:
            list of dicts containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, score threshold: {score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print(f"No documents found")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)        

In [18]:
rag_retriever

In [19]:
rag_retriever.retrieve("what are lists and dictionaries in python")

Retrieving documents for query: 'what are lists and dictionaries in python'
Top k: 5, score threshold: 0.0
Generating embeding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.06it/s]

Generated embedings with shape: (1, 384)


Retrieved 5 documents (after filtering)


[{'id': 'doc_964f9a95_28',
  'content': 'for x in new_dict: \n  print(x) \n \n#print all values in the dictionary  \n \nfor x in new_dict: \n  print(new_dict[x]) \n \n#loop through both keys and values \n \nfor x, y in my_dict.items(): \n  print(x, y)\nPython Cheat Sheet\n18\nWebsiteSetup.org - Python Cheat Sheet',
  'metadata': {'source': '..\\data\\pdf\\Python.pdf',
   'total_pages': 26,
   'doc_index': 28,
   'creator': 'Adobe InDesign 15.0 (Macintosh)',
   'content_length': 259,
   'page': 17,
   'file_type': 'pdf',
   'moddate': '2020-04-16T15:05:22+07:00',
   'creationdate': '2020-04-16T15:05:18+07:00',
   'trapped': '/False',
   'producer': 'Adobe PDF Library 15.0',
   'page_label': '18',
   'source_file': 'Python.pdf'},
  'similarity_score': 0.258173406124115,
  'distance': 0.741826593875885,
  'rank': 1},
 {'id': 'doc_6e7da0bb_25',
  'content': 'Convert Tuple to a List\nDictionaries\nHow to Create a Python Dictionary\nSince Tuples are immutable, you can’t change them. What you

In [21]:
rag_retriever.retrieve("what is primary key in sql table")

Retrieving documents for query: 'what is primary key in sql table'
Top k: 5, score threshold: 0.0
Generating embeding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.58it/s]

Generated embedings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_349a5c7c_94',
  'content': 'Keys\nPrimary Key\nExample 1 (MySQL)\nExample 2 (MySQL)\nIn relational databases, there is a concept of primary and foreign keys. In SQL tables, these are \nincluded as constraints, where a table can have a primary key, a foreign key, or both.\nA primary key allows each record in a table to be uniquely identified. There can only be one \nprimary key per table, and you can assign this constraint to any single or combination of columns. \nHowever, this means each value within this column(s) must be unique.\nTypically in a table, the primary key is an ID column, and is usually paired with the AUTO_\nINCREMENT keyword. This means the value increases automatically as new records are created.\nCREATE TABLE users (\nid int NOT NULL AUTO_INCREMENT,\nfirst_name varchar(255),\nlast_name varchar(255) NOT NULL,\naddress varchar(255),\nemail varchar(255),\nPRIMARY KEY (id)\n);\nALTER TABLE users\nADD PRIMARY KEY (first_name);\nCreate a new table and set the 

### RAG Pipeline - VectorDB to LLM Output Generation

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain.messages import HumanMessage, SystemMessage

In [24]:
class LLM:
    def __init__(self, model_name: str = "google_genai:gemini-2.5-flash-lite", api_key: str = None):
        """Initialize LLM

        Args:
            model_name (str): model to use. Default to google_genai:gemini-2.5-flash-lite
            api_key (str, optional): API key for the model. Defaults to None.
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GOOGLE_API_KEY")
        
        if not self.api_key:
            raise ValueError("API key is required for the model you want to use")
    
        self.llm = init_chat_model(model_name, temperature=0.1, max_tokens=1024)
        print(f"Initialized llm with model: {self.model_name}")
        
    def generate_response(self, query: str, context: str, max_length: int=500) -> str:
        """Generate response using retrieved context

        Args:
            query (str): user question
            context (str): retrieved documents context
            max_length (int, optional): maximum response length. Defaults to 500.

        Returns:
            str: generated response string
        """
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """Simple response generation without complex prompting

        Args:
            query (str): user question
            context (str): retrieved context

        Returns:
            str: response
        """
        simple_prompt = f"""Based on this context: {context}
        Question: {query}
        Answer:"""
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [25]:
try:
    llm = LLM(api_key=os.environ.get("GOOGLE_API_KEY"))
    print("Google gemini LLM initialized successfully")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set you api env variable to use the LLM and make request to correct model name")
    llm = None

Initialized llm with model: google_genai:gemini-2.5-flash-lite
Google gemini LLM initialized successfully


#### Integration vectordb context pipeline with LLM output

In [28]:
def rag_simnple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context document found for the query"
    
    response = llm.generate_response_simple(query, context)
    return response

In [29]:
query = "what is primary key in sql?"
answer = rag_simnple(query, rag_retriever, llm)
print(answer)

Retrieving documents for query: 'what is primary key in sql?'
Top k: 3, score threshold: 0.0
Generating embeding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 98.81it/s]

Generated embedings with shape: (1, 384)
Retrieved 1 documents (after filtering)


A primary key in SQL is a constraint that uniquely identifies each record in a table. It ensures that each value within the designated column(s) is unique and that there can only be one primary key per table. Typically, it's an ID column that is paired with the AUTO_INCREMENT keyword to automatically assign unique values to new records.


#### Enhanced RAG Pipeline features

In [31]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """RAG pipeline with extra features: 
    returns answer, sources, confidence_score, and optionally full context
    """
    results = retriever.retrieve(query, top_k, score_threshold=min_score)
    
    if not results:
        return {'answer': "No relevant context found", 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    response = llm.generate_response_simple(query, context)
    output = {
        'answer': response,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
    
    return output


result = rag_advanced("classes in python", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print(f"Answer: {result['answer']}")
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'classes in python'
Top k: 3, score threshold: 0.1
Generating embeding for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]


Generated embedings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Answer: Based on the provided context, here's an answer to the question "classes in python":

Classes in Python act as **blueprints for creating objects**. They define the structure and behavior that objects of that class will have. Almost every element in Python is an object, and classes are the fundamental way to define these objects, giving them their own methods and properties.

The context illustrates this with the `TestClass` example, which serves as a blueprint for creating objects that have a property named `z`. It also shows a more detailed `car` class with a constructor (`__init__`) to initialize object properties (like `color`, `doors`, `tires`) and methods (`brake`, `drive`) that define the object's actions.
Sources: [{'source': 'Python.pdf', 'page': 21, 'score': 0.24871093034744263, 'preview': 'Since Python is an object-oriented programming language almost every element of \nit is an object — 